# Retrieval Metrics
### Context Precision, Context Recall, Context Relevancy, Context Entity Recall — RAGAS

Corpus: `OWASP Top 10 for LLM Applications (2025)`. Same 8 ground-truth questions as the Generation Metrics notebook — retrieval metrics only need the *retrieved chunks* and the *reference* answer, not a generated answer, so this notebook stops right after retrieval.

## Step 1: Build the retriever (no generation LLM needed here)

In [1]:
#!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf ragas openai python-dotenv -q

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_9700\4025191060.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
incorrect startxref pointer(1)
parsing for Object Streams
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Ground truth — questions and reference answers
Same 8 questions used across all three RAGAS notebooks this week, so results are comparable side by side.

In [3]:
ground_truth = [
    {
        "question": "What is Prompt Injection, according to LLM01?",
        "reference": "A Prompt Injection Vulnerability occurs when user prompts alter the LLM's behavior or output in unintended ways. The input does not need to be human-readable, only something the model parses.",
    },
    {
        "question": "What kinds of sensitive information can an LLM application expose, according to LLM02?",
        "reference": "Personal identifiable information (PII), financial details, health records, confidential business data, security credentials, legal documents, and proprietary training methods or source code.",
    },
    {
        "question": "What real attack against a model hosted on Hugging Face is cited as an example of Supply Chain risk under LLM03?",
        "reference": "PoisonGPT: an attacker bypassed Hugging Face's safety features by directly tampering with a model's parameters to spread misinformation.",
    },
    {
        "question": "What is Excessive Agency, according to LLM06?",
        "reference": "Excessive Agency is the vulnerability where an LLM-based system is granted excessive functionality, permissions, or autonomy to call functions or tools, letting it take unintended or damaging actions.",
    },
    {
        "question": "What does System Prompt Leakage warn about, according to LLM07?",
        "reference": "System prompts can inadvertently contain sensitive information, such as credentials or internal business rules, that was not meant to be discovered; the system prompt should never be treated as a secret or used as a security control.",
    },
    {
        "question": "What security risks affect vectors and embeddings in RAG systems, according to LLM08?",
        "reference": "Weaknesses in how vectors and embeddings are generated, stored, or retrieved can be exploited to inject harmful content, manipulate outputs, or leak sensitive data -- for example, in a multi-tenant vector database, one group's embeddings could be retrieved in response to another group's queries.",
    },
    {
        "question": "What is the main cause of misinformation in LLMs, according to LLM09?",
        "reference": "Hallucination -- the LLM generates content that seems accurate but is fabricated, filling gaps in its training data using statistical patterns without truly understanding the content.",
    },
    {
        "question": "What is Unbounded Consumption, according to LLM10?",
        "reference": "A risk where an LLM application allows excessive, uncontrolled inference operations, which can lead to denial of service, runaway costs, or model theft through extraction and cloning attacks.",
    },
]

import pandas as pd
pd.DataFrame(ground_truth)

,question,reference
0,"What is Prompt Injection, according to LLM01?",A Prompt Injection Vulnerability occurs when u...
1,What kinds of sensitive information can an LLM...,"Personal identifiable information (PII), finan..."
2,What real attack against a model hosted on Hug...,PoisonGPT: an attacker bypassed Hugging Face's...
3,"What is Excessive Agency, according to LLM06?",Excessive Agency is the vulnerability where an...
4,"What does System Prompt Leakage warn about, ac...",System prompts can inadvertently contain sensi...
5,What security risks affect vectors and embeddi...,Weaknesses in how vectors and embeddings are g...
6,What is the main cause of misinformation in LL...,Hallucination -- the LLM generates content tha...
7,"What is Unbounded Consumption, according to LL...",A risk where an LLM application allows excessi...


## Step 3: Retrieve with top-k=3

In [4]:
for item in ground_truth:
    docs = vector_store.similarity_search(item["question"], k=3)
    item["contexts"] = [doc.page_content for doc in docs]

for item in ground_truth:
    print("Q:", item["question"])
    for i, ctx in enumerate(item["contexts"]):
        print(f"  [{i}]", ctx[:100].replace("\n", " "), "...")
    print()

Q: What is Prompt Injection, according to LLM01?
  [0] 1. Augmenting a Large Language Model with Retrieval-Augmented Generation and Fine- tuning 2. Astute  ...
  [1] Types of Prompt Injection Vulnerabilities .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . ...
  [2] Table of Contents Letter from the Project Leads  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .   ...

Q: What kinds of sensitive information can an LLM application expose, according to LLM02?
  [0] OWASP Top 10 for LLM Applications v2.0 7genai.owasp.org LLM02:2025 Sensitive Information Disclosure  ...
  [1] to interact safely with LLMs. They need to understand the risks of unintentionally providing sensiti ...
  [2] guardrails bypass, improper separation of privileges, etc. Even if the exact wording is not disclose ...

Q: What real attack against a model hosted on Hugging Face is cited as an example of Supply Chain risk under LLM03?
  [0] 2. Proprietary Algorithm Exposure Poorly configured model outputs can r

## Step 4: Wire RAGAS to an LLM judge
Same reasoning as the Generation Metrics notebook: retrieval scoring also needs a reliable instruction-follower, so the judge is `gpt-4o-mini`. Retrieval itself stays fully local on Ollama + FAISS — only the *judge* is cloud-based.

In [5]:
import sys, types

_stub = types.ModuleType("langchain_community.chat_models.vertexai")


class ChatVertexAI:
    pass


_stub.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _stub

from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from ragas.llms.base import llm_factory

client = AsyncOpenAI()
judge_llm = llm_factory("gpt-4o-mini", client=client)

print("Judge ready.")

Judge ready.


## Step 5: Context Precision — are the retrieved chunks relevant?
For each retrieved chunk, the judge decides whether it's actually useful for answering the question, and precision is computed with position-weighting (a relevant chunk ranked #1 counts more than one ranked #3).

In [6]:
from ragas.metrics.collections import ContextPrecision

context_precision_metric = ContextPrecision(llm=judge_llm)

for item in ground_truth:
    result = await context_precision_metric.ascore(
        user_input=item["question"],
        reference=item["reference"],
        retrieved_contexts=item["contexts"],
    )
    item["context_precision"] = result.value

pd.DataFrame(ground_truth)[["question", "context_precision"]]

,question,context_precision
0,"What is Prompt Injection, according to LLM01?",0.000000
1,What kinds of sensitive information can an LLM...,1.000000
2,What real attack against a model hosted on Hug...,0.333333
3,"What is Excessive Agency, according to LLM06?",0.000000
4,"What does System Prompt Leakage warn about, ac...",0.500000
5,What security risks affect vectors and embeddi...,0.833333
6,What is the main cause of misinformation in LL...,0.500000
7,"What is Unbounded Consumption, according to LL...",0.000000


## Step 6: Context Recall — did we retrieve everything needed?
The reference answer is decomposed into sentences, and each one is checked for support in the retrieved chunks. A low score means relevant documents exist somewhere in the corpus but weren't retrieved.

In [7]:
from ragas.metrics.collections import ContextRecall

context_recall_metric = ContextRecall(llm=judge_llm)

for item in ground_truth:
    result = await context_recall_metric.ascore(
        user_input=item["question"],
        reference=item["reference"],
        retrieved_contexts=item["contexts"],
    )
    item["context_recall"] = result.value

pd.DataFrame(ground_truth)[["question", "context_recall"]]

,question,context_recall
0,"What is Prompt Injection, according to LLM01?",1.0
1,What kinds of sensitive information can an LLM...,1.0
2,What real attack against a model hosted on Hug...,1.0
3,"What is Excessive Agency, according to LLM06?",0.0
4,"What does System Prompt Leakage warn about, ac...",0.5
5,What security risks affect vectors and embeddi...,1.0
6,What is the main cause of misinformation in LL...,1.0
7,"What is Unbounded Consumption, according to LL...",1.0


## Step 7: Context Relevancy — signal-to-noise within the chunks
Works at sentence level *inside* each retrieved chunk, rather than chunk level — a chunk can be "relevant" overall (Context Precision) while still being mostly padding around one useful sentence.

In [8]:
from ragas.metrics.collections import ContextRelevance

context_relevancy_metric = ContextRelevance(llm=judge_llm)

for item in ground_truth:
    result = await context_relevancy_metric.ascore(
        user_input=item["question"],
        retrieved_contexts=item["contexts"],
    )
    item["context_relevancy"] = result.value

pd.DataFrame(ground_truth)[["question", "context_relevancy"]]

,question,context_relevancy
0,"What is Prompt Injection, according to LLM01?",1.00
1,What kinds of sensitive information can an LLM...,1.00
2,What real attack against a model hosted on Hug...,0.75
3,"What is Excessive Agency, according to LLM06?",0.00
4,"What does System Prompt Leakage warn about, ac...",0.25
5,What security risks affect vectors and embeddi...,1.00
6,What is the main cause of misinformation in LL...,1.00
7,"What is Unbounded Consumption, according to LL...",0.00


## Step 8: Context Entity Recall — did we retrieve the key facts?
A non-LLM-adjacent, NER-based check: extracts named entities (dates, names, numbers) from the reference and from the retrieved chunks, and measures overlap. Especially useful for factoid questions where a topically-similar chunk can still be missing the one number that matters.

In [9]:
from ragas.metrics.collections import ContextEntityRecall

context_entity_recall_metric = ContextEntityRecall(llm=judge_llm)

for item in ground_truth:
    result = await context_entity_recall_metric.ascore(
        reference=item["reference"],
        retrieved_contexts=item["contexts"],
    )
    item["context_entity_recall"] = result.value

pd.DataFrame(ground_truth)[["question", "context_entity_recall"]]

,question,context_entity_recall
0,"What is Prompt Injection, according to LLM01?",0.000000
1,What kinds of sensitive information can an LLM...,0.777778
2,What real attack against a model hosted on Hug...,1.000000
3,"What is Excessive Agency, according to LLM06?",0.000000
4,"What does System Prompt Leakage warn about, ac...",0.000000
5,What security risks affect vectors and embeddi...,0.000000
6,What is the main cause of misinformation in LL...,0.500000
7,"What is Unbounded Consumption, according to LL...",0.000000


## Step 9: Scorecard — top-k=3 retrieval

In [10]:
score_cols = [
    "context_precision", "context_recall", "context_relevancy", "context_entity_recall"
]
scorecard = pd.DataFrame(ground_truth)[["question"] + score_cols]
scorecard.loc["mean", "question"] = ""
scorecard.loc["mean", score_cols] = scorecard[score_cols].mean()
scorecard

,question,context_precision,context_recall,context_relevancy,context_entity_recall
0,"What is Prompt Injection, according to LLM01?",0.000000,1.0000,1.000,0.000000
1,What kinds of sensitive information can an LLM...,1.000000,1.0000,1.000,0.777778
2,What real attack against a model hosted on Hug...,0.333333,1.0000,0.750,1.000000
3,"What is Excessive Agency, according to LLM06?",0.000000,0.0000,0.000,0.000000
4,"What does System Prompt Leakage warn about, ac...",0.500000,0.5000,0.250,0.000000
5,What security risks affect vectors and embeddi...,0.833333,1.0000,1.000,0.000000
6,What is the main cause of misinformation in LL...,0.500000,1.0000,1.000,0.500000
7,"What is Unbounded Consumption, according to LL...",0.000000,1.0000,0.000,0.000000
mean,,0.395833,0.8125,0.625,0.284722


In [11]:
# Production targets from the slide deck's scorecard
targets = {
    "context_precision": 0.75,
    "context_recall": 0.70,
    "context_relevancy": 0.60,
    "context_entity_recall": 0.70,
}

means = scorecard.loc["mean", score_cols]
for metric, target in targets.items():
    mean_score = means[metric]
    verdict = "PASS" if mean_score >= target else "BELOW MINIMUM"
    print(f"{metric:<22} mean={mean_score:.2f}  minimum={target:.2f}  -> {verdict}")

context_precision      mean=0.40  minimum=0.75  -> BELOW MINIMUM
context_recall         mean=0.81  minimum=0.70  -> PASS
context_relevancy      mean=0.62  minimum=0.60  -> PASS
context_entity_recall  mean=0.28  minimum=0.70  -> BELOW MINIMUM


## Step 10: Deliberately weak retrieval — top-k=1
Same questions, same index — just retrieve a single chunk instead of three, and recompute every metric. If the metrics are doing their job, Context Precision and Recall should visibly drop.

In [12]:
for item in ground_truth:
    weak_docs = vector_store.similarity_search(item["question"], k=1)
    item["weak_contexts"] = [doc.page_content for doc in weak_docs]

for item in ground_truth:
    weak_precision = await context_precision_metric.ascore(
        user_input=item["question"],
        reference=item["reference"],
        retrieved_contexts=item["weak_contexts"],
    )
    weak_recall = await context_recall_metric.ascore(
        user_input=item["question"],
        reference=item["reference"],
        retrieved_contexts=item["weak_contexts"],
    )
    weak_relevancy = await context_relevancy_metric.ascore(
        user_input=item["question"],
        retrieved_contexts=item["weak_contexts"],
    )
    weak_entity_recall = await context_entity_recall_metric.ascore(
        reference=item["reference"],
        retrieved_contexts=item["weak_contexts"],
    )
    item["weak_context_precision"] = weak_precision.value
    item["weak_context_recall"] = weak_recall.value
    item["weak_context_relevancy"] = weak_relevancy.value
    item["weak_context_entity_recall"] = weak_entity_recall.value

## Step 11: Compare top-k=3 vs top-k=1 side by side

In [13]:
comparison = pd.DataFrame(ground_truth)[["question"] + score_cols]
for col in score_cols:
    comparison[f"{col}_k1"] = pd.DataFrame(ground_truth)[f"weak_{col}"]
comparison.loc["mean"] = comparison.drop(columns=["question"]).mean()
comparison.loc["mean", "question"] = ""

print("k=3 vs k=1 mean scores:")
for col in score_cols:
    k3_mean = comparison.loc["mean", col]
    k1_mean = comparison.loc["mean", f"{col}_k1"]
    print(f"{col:<24} k=3: {k3_mean:.2f}   k=1: {k1_mean:.2f}   drop: {k3_mean - k1_mean:+.2f}")

k=3 vs k=1 mean scores:
context_precision        k=3: 0.40   k=1: 0.25   drop: +0.15
context_recall           k=3: 0.81   k=1: 0.31   drop: +0.50
context_relevancy        k=3: 0.62   k=1: 0.38   drop: +0.25
context_entity_recall    k=3: 0.28   k=1: 0.12   drop: +0.16


## Try it yourself
1. Try `top-k=10` instead of `k=1` — do Context Precision and Context Relevancy start dropping too, now from too much noise instead of too little coverage?
2. Pick the question with the lowest Context Entity Recall and print its retrieved chunks — is the missing entity genuinely absent from the corpus, or just not retrieved?
3. Increase `chunk_size` to 2000 and rerun Context Relevancy — bigger chunks usually raise Context Precision (more likely to contain *something* relevant) while lowering Context Relevancy (more padding around it).